# Shafaa: 2-, 4-, and 8-second planning horizons

Branch: `shafaa`, based on Tianzi v2 commit `3638812`.
Use **Runtime → Change runtime type → T4 GPU**, then **Run all**.

This notebook runs three full attempts on scene 100038, seed 0, and displays their videos.
Allow roughly 15–25 minutes on a T4. This is a demonstration, not a benchmark conclusion.

Only the world-model prediction horizon changes (4/8/16 blocks). Every variant samples
16 candidates of 16 blocks, uses terminal squared latent distance and the original
goal-switching/action rules, and executes one block (5 control steps / 0.5 seconds).
The original policy, world model and official 8-second subgoal images stay unchanged.
No goal-conditioned v2 trained actor, training, optimizer, or privileged planner input is used.


In [ ]:
import os, subprocess, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("installing system libraries", flush=True)
    env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
    subprocess.check_call(["apt-get", "update", "-qq"], env=env)
    subprocess.check_call(
        ["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "libgles2", "libglib2.0-0", "ffmpeg"],
        env=env,
    )

def missing(module):
    try:
        __import__(module)
        return False
    except Exception:
        return True

packages = []
if missing("mujoco"):
    packages.append("mujoco>=3.3,<4")
if missing("imageio_ffmpeg"):
    packages.append("imageio[ffmpeg]>=2.30")
if missing("huggingface_hub"):
    packages.append("huggingface_hub>=0.26")
if missing("matplotlib"):
    packages.append("matplotlib")
if packages:
    print("installing", ", ".join(packages), flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *packages])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is visible. In Colab: Runtime → Change runtime type → GPU, then Runtime → Run all.")
print("cuda", torch.cuda.get_device_name(0), flush=True)


In [ ]:
import os, sys, tarfile
from pathlib import Path
os.environ["MUJOCO_GL"] = "egl"
os.environ["HF_HUB_DISABLE_XET"] = "1"

def find_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "assignment" / "training" / "planning" / "agent.py").exists():
            return cand
    return None

ROOT = find_root()
if ROOT is None:
    dest = Path("/content") if Path("/content").exists() else Path("/tmp")
    ROOT = dest / "world-model-gym"
    marker = ROOT / "assignment" / "training" / "planning" / "agent.py"
    if not marker.exists():
        print("downloading code", flush=True)
        from huggingface_hub import hf_hub_download
        archive = hf_hub_download("YongYong/cis6280-stack-world-models-v2", "code/world-model-gym.tar", repo_type="model")
        with tarfile.open(archive) as tar:
            try:
                tar.extractall(dest, filter="data")
            except TypeError:
                tar.extractall(dest)
    if not marker.exists():
        raise FileNotFoundError(f"code download did not produce {marker}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
hand = ROOT / "assets" / "sharpa" / "wave_01" / "right_sharpa_wave" / "right_sharpa_wave.xml"
if not hand.exists():
    print("downloading hand assets", flush=True)
    __import__("subprocess").check_call([sys.executable, "scripts/fetch_assets.py"])
import assignment.training.planning.agent
print("code", ROOT, flush=True)

from assignment.training.data.task import StackTaskEnv, TaskConfig
_env = StackTaskEnv(TaskConfig(target_count=4, horizon=10, terminate_on_ground=False), 100038)
_env.reset(("init", 100038))
_env.env.resolution = 128
# render() returns the (front, side) camera tuple, each of shape (128, 128, 3).
_frames = list(_env.env.render())
_env.close()
print("render", _frames[0].shape, flush=True)

In [ ]:
# Download original inputs; pass the WM path explicitly instead of editing policy.pt.
import shutil
from huggingface_hub import hf_hub_download
CKPT = Path('/content/horizon-inputs')
CKPT.mkdir(parents=True, exist_ok=True)
WM, POLICY, GOALS = CKPT/'model.pt', CKPT/'policy.pt', CKPT/'goals.npz'
for repo, name, dest, kind in [
    ('YongYong/cis6280-stack-world-models-v2', 'jepa/smwm-c3/model.pt', WM, 'model'),
    ('YongYong/cis6280-stack-world-models-v2', 'policy/df-smwm-c1-sigma04/policy.pt', POLICY, 'model'),
    ('YongYong/cis6280-stack-datasets-v2', 'goals/scripted-val100-every80/goals.npz', GOALS, 'dataset'),
]:
    shutil.copyfile(hf_hub_download(repo, name, repo_type=kind), dest)
print('Original frozen inputs ready.')


In [ ]:
# Install the exact experiment source embedded in this notebook.
# This writes code only; the downloaded model and goals files are never edited.
import importlib, json, subprocess, sys
from pathlib import Path
sources = json.loads('{"assignment/training/planning/agent.py": "\\"\\"\\"Closed-loop policy agent and goal-conditioned planning agent.\\"\\"\\"\\nimport contextlib\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nfrom assignment.training.data.cache import RAW_ACTION_DIM, STRIDE\\nfrom assignment.training.training.jepa import amp_dtype\\nfrom assignment.training.training.policy import load_policy\\n\\n\\nclass LatentPolicyAgent:\\n    \\"\\"\\"Closed-loop controller that samples action chunks from the diffusion-forcing policy.\\n\\n    Call step(frame) at every block boundary (every STRIDE=5 control steps) with the tiled\\n    (128, 256, 3) uint8 RGB frame; returns the (STRIDE, 25) float32 raw actions in [-1, 1]\\n    for that block.\\n    \\"\\"\\"\\n\\n    def __init__(self, policy_path, device=\'cuda\', generate=None, execute=1, n_steps=10, lag=0, seed=0,\\n                 world_model=None):\\n        from assignment.training.training.jepa import load_model\\n        self.policy, info = load_policy(policy_path, device)\\n        wm_path = Path(world_model or info[\'encoder\'])\\n        if not wm_path.exists():\\n            raise FileNotFoundError(\\n                f\\"World-model checkpoint not found at \'{wm_path}\'. Pass world_model=\'path/to/model.pt\' \\"\\n                f\\"or set ckpt[\'info\'][\'encoder\'] in \'{policy_path}\' to your local world-model file.\\"\\n            )\\n        self.encoder, _ = load_model(wm_path, device)\\n        self.device, self.info = str(device), info\\n        self._device_type = torch.device(device).type\\n        self._amp_dtype = amp_dtype()\\n        self.generate = generate or self.policy.cfg.horizon\\n        self.execute, self.n_steps, self.lag = execute, n_steps, lag\\n        if not 1 <= execute <= self.generate <= self.policy.cfg.horizon:\\n            raise ValueError(\'need 1 <= execute <= generate <= horizon\')\\n        self.z_mean = torch.tensor(info[\'z_mean\'], device=device)\\n        self.z_std = torch.tensor(info[\'z_std\'], device=device)\\n        self.a_mean = np.asarray(info[\'action_mean\'], np.float32)\\n        self.a_std = np.asarray(info[\'action_std\'], np.float32)\\n        self.generator = torch.Generator(device).manual_seed(seed)\\n        self.reset()\\n\\n    def _autocast(self):\\n        if self._device_type == \'cuda\':\\n            return torch.autocast(\'cuda\', self._amp_dtype)\\n        return contextlib.nullcontext()\\n\\n    def reset(self, seed=None):\\n        \\"\\"\\"Start an episode. Passing a seed (e.g. the scene seed) makes the episode\'s samples\\n        independent of which episodes this agent ran before.\\"\\"\\"\\n        if seed is not None:\\n            self.generator.manual_seed(int(seed))\\n        self.history, self.queue = [], []\\n\\n    @torch.no_grad()\\n    def step(self, frame):\\n        x = torch.from_numpy(np.ascontiguousarray(frame)).to(self.device)[None, None]\\n        with self._autocast():\\n            z = self.encoder.encode(x)[0, 0].float()\\n        self.history = (self.history+[(z-self.z_mean)/self.z_std])[-self.policy.cfg.context:]\\n        if not self.queue:\\n            ctx = [self.history[0]]*(self.policy.cfg.context-len(self.history))+self.history\\n            blocks = self.policy.sample(torch.stack(ctx)[None], self.generate, self.n_steps, self.lag,\\n                                        self.generator)[0, :self.execute].cpu().numpy()\\n            raw = np.clip(blocks*self.a_std+self.a_mean, -1, 1).reshape(-1, STRIDE, RAW_ACTION_DIM)\\n            self.queue = list(raw)\\n        return self.queue.pop(0)\\n\\n\\nclass GoalPlanningAgent(LatentPolicyAgent):\\n    \\"\\"\\"Test-time planner that scores candidate policy chunks with the latent world model.\\n\\n    At each replanning step:\\n      1. Encode the current tiled RGB frame I_k into a latent z_k in R^192 and advance the\\n         subgoal index m whenever ||z_k - g_m||_2^2 < tau * ||g_m - g_{m-1}||_2^2.\\n      2. Draw `candidates` chunks of `generate` action blocks from the policy.\\n      3. Roll each candidate chunk forward in the world model from the last C<=3 real frame\\n         latents (and the action blocks executed between them) to predict latents z_hat.\\n      4. Execute the first `execute` blocks of the candidate whose final predicted latent\\n         z_hat_{k+planning_horizon} is closest in squared L2 distance to the active subgoal g_m.\\n\\n    planning_horizon counts 0.5-second blocks, defaults to generate, and only\\n    truncates WM prediction. The policy always samples generate complete blocks.\\n\\n    Goals are an ordered subgoal sequence; the index advances monotonically\\n    while ||z - g_m||_2^2 < tau * ||g_m - g_{m-1}||_2^2, with g_{-1} the latent of init_frame.\\n    Call reset(goal_frames, init_frame, seed) per episode and step(frame) every STRIDE steps.\\n    \\"\\"\\"\\n\\n    def __init__(self, policy_path, device=\'cuda\', candidates=16, tau=.25,\\n                 planning_horizon=None, **kwargs):\\n        super().__init__(policy_path, device, **kwargs)\\n        # Policy generation and WM prediction are independent. Keep sampling the\\n        # usual full plans while scoring only their requested future prefix.\\n        horizon = self.generate if planning_horizon is None else planning_horizon\\n        if isinstance(horizon, bool) or not isinstance(horizon, (int, np.integer)) or not self.execute <= horizon <= self.generate:\\n            raise ValueError(\'need an integer execute <= planning_horizon <= generate\')\\n        self.planning_horizon = int(horizon)\\n        self.candidates, self.tau = candidates, tau\\n        self.goals = None\\n\\n    def reset(self, goal_frames=None, init_frame=None, seed=None):\\n        if goal_frames is not None and init_frame is None:\\n            raise ValueError(\'init_frame is required with goal_frames\')\\n        super().reset(seed)\\n        self.frames, self.executed, self.index = [], [], 0\\n        self.last_distance = self.goals = self.gap = None\\n        if goal_frames is not None:\\n            self.goals = self._encode(np.asarray(goal_frames))\\n            prev = torch.cat([self._encode(np.asarray(init_frame)[None]), self.goals[:-1]])\\n            self.gap = (self.goals-prev).square().sum(-1)\\n\\n    def goal_index(self, z):\\n        while self.index < len(self.goals)-1 and \\\\\\n                (z-self.goals[self.index]).square().sum() < self.tau*self.gap[self.index]:\\n            self.index += 1\\n        return self.index\\n\\n    @torch.no_grad()\\n    def _encode(self, frames):\\n        x = torch.from_numpy(np.ascontiguousarray(frames)).to(self.device)[:, None]\\n        with self._autocast():\\n            return self.encoder.encode(x)[:, 0].float()\\n\\n    @torch.no_grad()\\n    def step(self, frame):\\n        if self.goals is None:\\n            raise RuntimeError(\'Call agent.reset(goal_frames, init_frame) before step().\')\\n        z = self._encode(frame[None])[0]\\n        ctx = self.encoder.cfg.context\\n        self.frames = (self.frames+[z])[-ctx:]\\n        self.history = (self.history+[(z-self.z_mean)/self.z_std])[-self.policy.cfg.context:]\\n        goal = self.goals[self.goal_index(z)]\\n        if not self.queue:\\n            pctx = [self.history[0]]*(self.policy.cfg.context-len(self.history))+self.history\\n            n = self.candidates\\n            blocks = self.policy.sample(torch.stack(pctx)[None].expand(n, -1, -1), self.generate, self.n_steps,\\n                                        self.lag, self.generator)\\n            best = 0\\n            if n > 1:\\n                # Pair the C-1 earlier context frames with the C-1 blocks executed after them,\\n                # and the current frame with the n candidate chunks of length `generate`.\\n                past = self.executed[len(self.executed)-len(self.frames)+1:] if len(self.frames) > 1 else []\\n                future = blocks[:, :self.planning_horizon]\\n                acts = torch.cat([torch.stack(past)[None].expand(n, -1, -1), future], 1) if past else future\\n                with self._autocast():\\n                    pred = self.encoder.rollout(torch.stack(self.frames)[None].expand(n, -1, -1),\\n                                                self.encoder.action_encoder(acts), self.planning_horizon)\\n                self.last_distance = (pred[:, -1].float()-goal).square().sum(-1)\\n                best = int(self.last_distance.argmin())\\n            raw = np.clip(blocks[best, :self.execute].cpu().numpy()*self.a_std+self.a_mean, -1, 1)\\n            self.queue = list(raw.reshape(-1, STRIDE, RAW_ACTION_DIM))\\n        block = self.queue.pop(0)\\n        self.executed.append(torch.from_numpy((block.reshape(-1)-self.a_mean)/self.a_std).to(self.device))\\n        return block\\n", "assignment/planning/horizon_experiment.py": "\\"\\"\\"Frozen-model 2/4/8-second planning experiment, usable from Colab or CLI.\\n\\nOnly WM prediction length changes. Policy generation stays at 16 blocks;\\nofficial terminal-L2 scoring, action handling and goal switching are retained.\\n\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport platform\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nfrom assignment.training.planning.agent import GoalPlanningAgent\\n\\nHORIZONS = {2: 4, 4: 8, 8: 16}\\n\\n\\ndef digest(path):\\n    h = hashlib.sha256()\\n    with open(path, \'rb\') as source:\\n        for block in iter(lambda: source.read(1024 * 1024), b\'\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef write_json(path, value):\\n    path = Path(path)\\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\\\n\')\\n    temporary.replace(path)\\n\\n\\ndef summarize(rows):\\n    result = []\\n    for seconds in sorted({r[\'planning_seconds\'] for r in rows}):\\n        group = [r for r in rows if r[\'planning_seconds\'] == seconds]\\n        timings = [t for r in group for t in r[\'decision_ms\']]\\n        result.append(dict(planning_seconds=seconds, episodes=len(group),\\n                           mean_peak_boxes=float(np.mean([r[\'peak_boxes\'] for r in group])),\\n                           mean_final_boxes=float(np.mean([r[\'final_boxes\'] for r in group])),\\n                           success_rate=float(np.mean([r[\'success\'] for r in group])),\\n                           mean_decision_ms=float(np.mean(timings)),\\n                           p95_decision_ms=float(np.percentile(timings, 95))))\\n    return result\\n\\n\\ndef run_episode(agent, goal_frames, init_frame, scene, sample_seed, seconds, horizon, folder):\\n    # Simulator task and action loop match the official evaluation. Privileged\\n    # state is used only by the evaluator for metrics, never by the planner.\\n    from assignment.training.data.cache import STRIDE\\n    from assignment.training.data.recording import write_video\\n    from assignment.training.data.task import StackTaskEnv, TaskConfig\\n    agent.reset(goal_frames, init_frame, seed=int(scene) + 1_000_000 * int(sample_seed))\\n    env = StackTaskEnv(TaskConfig(target_count=4, horizon=horizon, terminate_on_ground=False), int(scene))\\n    frames, times, peak, info, success = [], [], 0, {}, False\\n    start = time.perf_counter()\\n    try:\\n        env.reset((\'init\', int(scene)))\\n        env.env.resolution = 128\\n        with (folder / \'decisions.jsonl\').open(\'w\') as trace:\\n            for step in range(horizon):\\n                if step % STRIDE == 0:\\n                    frame = np.concatenate(list(env.env.render()), axis=1)\\n                    if agent._device_type == \'cuda\':\\n                        torch.cuda.synchronize(agent.device)\\n                    t0 = time.perf_counter()\\n                    block = agent.step(frame)\\n                    if agent._device_type == \'cuda\':\\n                        torch.cuda.synchronize(agent.device)\\n                    elapsed = 1000 * (time.perf_counter() - t0)\\n                    times.append(elapsed)\\n                    if block.shape != (5, 25) or block.dtype != np.float32 or not np.isfinite(block).all() or np.abs(block).max() > 1:\\n                        raise ValueError(\'Invalid planner action block\')\\n                    frames.append(np.concatenate([frame, goal_frames[agent.index]], axis=0))\\n                    trace.write(json.dumps(dict(control_step=step, goal_index=agent.index,\\n                                                decision_ms=elapsed,\\n                                                candidate_costs=agent.last_distance.cpu().tolist())) + \'\\\\n\')\\n                    trace.flush()\\n                    if step % 200 == 0:\\n                        print(f\'{seconds}s / scene {scene} / seed {sample_seed}: step {step}/{horizon}, peak {peak}\', flush=True)\\n                _, _, terminated, truncated, info = env.step(block[step % STRIDE])\\n                peak = max(peak, int(info[\'placed\']))\\n                if terminated or truncated:\\n                    success = bool(info.get(\'success\', False))\\n                    break\\n    finally:\\n        env.close()\\n    video = folder / \'video.mp4\'\\n    write_video(video, [f[None] for f in frames], STRIDE)\\n    row = dict(planning_seconds=seconds, planning_blocks=HORIZONS[seconds], scene=int(scene),\\n               sample_seed=int(sample_seed), control_steps=step + 1,\\n               peak_boxes=peak, final_boxes=int(info.get(\'count\', 0)), success=success,\\n               decision_ms=times, wall_seconds=time.perf_counter() - start, video=str(video))\\n    write_json(folder / \'episode.json\', row)  # Completion marker written last.\\n    print(f\'Finished {seconds}s: peak {peak}, final {row[\\"final_boxes\\"]}, success {success}\', flush=True)\\n    return row\\n\\n\\ndef run_experiment(policy, world_model, goals, out, scenes=(100038,), sample_seeds=(0,),\\n                   planning_seconds=(2, 4, 8), episode_steps=1795, device=\'cuda\', resume=True):\\n    \\"\\"\\"Run a small paired comparison. Short episodes are implementation checks only.\\"\\"\\"\\n    from assignment.training.data.signals import scene_split\\n    scenes, sample_seeds, planning_seconds = list(scenes), list(sample_seeds), list(planning_seconds)\\n    if not scenes or not sample_seeds or not planning_seconds:\\n        raise ValueError(\'Scenes, sample seeds and horizons must be nonempty\')\\n    if any(len(v) != len(set(v)) for v in (scenes, sample_seeds, planning_seconds)):\\n        raise ValueError(\'Duplicate scenes, seeds or horizons\')\\n    if any(s not in HORIZONS for s in planning_seconds) or not 1 <= episode_steps <= 1795:\\n        raise ValueError(\'Use 2/4/8 seconds and an episode length from 1 to 1795\')\\n    if any(scene_split(int(s)) != \'val\' for s in scenes):\\n        raise ValueError(\'All scenes must be in the official validation split\')\\n    with np.load(goals) as source:\\n        data = {k: source[k] for k in source.files}\\n    index = {int(s): i for i, s in enumerate(data[\'seeds\'])}\\n    if any(int(s) not in index for s in scenes):\\n        raise ValueError(\'A requested scene is missing from official goals\')\\n    inputs = {name: dict(path=str(Path(path).resolve()), sha256=digest(path))\\n              for name, path in dict(policy=policy, world_model=world_model, goals=goals).items()}\\n    # Hash the actual source even when Colab installs it into the downloaded code tree.\\n    source_paths = [Path(__file__), Path(__file__).parents[1] / \'training/planning/agent.py\']\\n    manifest = dict(protocol=\'shafaa-horizon-v1\', inputs=inputs,\\n                    sources={p.name: digest(p) for p in source_paths},\\n                    scenes=scenes, sample_seeds=sample_seeds, planning_seconds=planning_seconds,\\n                    episode_steps=episode_steps, smoke_only=episode_steps != 1795,\\n                    generate=16, candidates=16, execute=1, n_steps=10, lag=0, tau=.25,\\n                    scoring=\'terminal_squared_L2\', prediction_action_handling=\'official_baseline\',\\n                    device=str(device), python=platform.python_version(), torch=torch.__version__,\\n                    numpy=np.__version__, gpu=(torch.cuda.get_device_name(device) if str(device).startswith(\'cuda\') else None))\\n    out = Path(out)\\n    out.mkdir(parents=True, exist_ok=True)\\n    receipt = out / \'run.json\'\\n    if receipt.exists():\\n        if not resume or json.loads(receipt.read_text()) != manifest:\\n            raise ValueError(\'Output exists with different settings/source. Use a fresh output folder.\')\\n    else:\\n        if any(out.iterdir()):\\n            raise ValueError(\'Output folder is nonempty without a run manifest\')\\n        write_json(receipt, manifest)\\n    agent = GoalPlanningAgent(str(policy), device=device, world_model=str(world_model),\\n                              candidates=16, generate=16, execute=1, n_steps=10, lag=0, tau=.25)\\n    agent.policy.eval().requires_grad_(False)\\n    agent.encoder.eval().requires_grad_(False)\\n    rows = []\\n    for seconds in planning_seconds:\\n        agent.planning_horizon = HORIZONS[seconds]\\n        for scene in scenes:\\n            i = index[int(scene)]\\n            goal_frames = data[\'sub_frames\'][i, :int(data[\'sub_count\'][i])]\\n            for seed in sample_seeds:\\n                folder = out / f\'horizon-{seconds}s\' / f\'scene-{scene}.sample-{seed}\'\\n                folder.mkdir(parents=True, exist_ok=True)\\n                marker = folder / \'episode.json\'\\n                if marker.exists() and (folder / \'video.mp4\').is_file():\\n                    row = json.loads(marker.read_text())\\n                    print(f\'Reusing completed {seconds}s / {scene} / {seed}\', flush=True)\\n                else:\\n                    row = run_episode(agent, goal_frames, data[\'init_frames\'][i], scene, seed,\\n                                      seconds, episode_steps, folder)\\n                rows.append(row)\\n                write_json(out / \'summary.json\', dict(smoke_only=episode_steps != 1795, groups=summarize(rows)))\\n    for name, path in dict(policy=policy, world_model=world_model, goals=goals).items():\\n        if digest(path) != inputs[name][\'sha256\']:\\n            raise RuntimeError(f\'Input file changed during experiment: {name}\')\\n    print(json.dumps(summarize(rows), indent=2), flush=True)\\n    return rows\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\'--policy\', required=True)\\n    p.add_argument(\'--world-model\', required=True)\\n    p.add_argument(\'--goals\', required=True)\\n    p.add_argument(\'--out\', required=True)\\n    p.add_argument(\'--scenes\', type=int, nargs=\'+\', default=[100038])\\n    p.add_argument(\'--sample-seeds\', type=int, nargs=\'+\', default=[0])\\n    p.add_argument(\'--planning-seconds\', type=int, nargs=\'+\', default=[2, 4, 8])\\n    p.add_argument(\'--episode-steps\', type=int, default=1795)\\n    p.add_argument(\'--device\', default=\'cuda\')\\n    args = vars(p.parse_args())\\n    run_experiment(**args)\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "tests/test_planning_horizon.py": "\\"\\"\\"Check horizon selection, real context alignment and frozen-baseline equivalence.\\"\\"\\"\\nimport contextlib\\nimport types\\nimport unittest\\nfrom unittest import mock\\n\\nimport numpy as np\\nimport torch\\n\\nfrom assignment.training.planning.agent import GoalPlanningAgent, LatentPolicyAgent\\n\\n\\nclass Policy:\\n    cfg = types.SimpleNamespace(context=1, horizon=16)\\n\\n    def __init__(self):\\n        self.calls = []\\n\\n    def sample(self, context, generate, n_steps, lag, generator):\\n        self.calls.append((context.clone(), generate))\\n        blocks = torch.zeros(context.shape[0], generate, 125)\\n        # Candidate 0 reaches goal=1 at block 4 and then moves away.\\n        # Candidate 1 reaches it at block 16. Their first actions differ.\\n        blocks[0, :, 0] = .25\\n        blocks[1, 0, 0] = -.25\\n        blocks[1, 1:, 0] = 1.25 / 15\\n        return blocks\\n\\n\\nclass Encoder:\\n    cfg = types.SimpleNamespace(context=3)\\n\\n    def __init__(self):\\n        self.calls = []\\n\\n    def action_encoder(self, acts):\\n        return acts\\n\\n    def rollout(self, frames, acts, steps):\\n        self.calls.append((frames.clone(), acts.clone(), steps))\\n        prefix = frames.shape[1] - 1\\n        return frames[:, -1:] + acts[:, prefix:prefix + steps, :1].cumsum(1)\\n\\n\\ndef setup_agent(agent):\\n    agent.device = \'cpu\'\\n    agent._device_type = \'cpu\'\\n    agent._autocast = contextlib.nullcontext\\n    agent.generator = torch.Generator().manual_seed(0)\\n    agent.policy, agent.encoder = Policy(), Encoder()\\n    agent.generate, agent.execute, agent.n_steps, agent.lag = 16, 1, 10, 0\\n    agent.z_mean, agent.z_std = torch.zeros(1), torch.ones(1)\\n    agent.a_mean, agent.a_std = np.zeros(125, np.float32), np.ones(125, np.float32)\\n    agent._encode = lambda frames: torch.from_numpy(np.asarray(frames)).float().reshape(len(frames), -1)\\n    LatentPolicyAgent.reset(agent)\\n\\n\\ndef fake_base_init(self, *args, **kwargs):\\n    setup_agent(self)\\n\\n\\ndef make_agent(blocks=None, cls=GoalPlanningAgent):\\n    with mock.patch.object(LatentPolicyAgent, \'__init__\', fake_base_init):\\n        if cls is GoalPlanningAgent:\\n            agent = cls(\'unused\', candidates=2, planning_horizon=blocks)\\n        else:\\n            agent = cls(\'unused\', candidates=2)\\n    agent.reset(np.array([[1]], np.uint8), np.array([0], np.uint8), seed=100038)\\n    return agent\\n\\n\\nclass HorizonTests(unittest.TestCase):\\n    def test_requested_prefix_only_and_full_policy_generation(self):\\n        for blocks in (4, 8, 16):\\n            agent = make_agent(blocks)\\n            for _ in range(4):\\n                action = agent.step(np.array([0], np.uint8))\\n                self.assertEqual(action.shape, (5, 25))\\n                self.assertEqual(action.dtype, np.float32)\\n                self.assertTrue(np.isfinite(action).all())\\n                self.assertLessEqual(float(np.abs(action).max()), 1)\\n            self.assertTrue(all(generate == 16 for _, generate in agent.policy.calls))\\n            for frames, acts, steps in agent.encoder.calls:\\n                self.assertEqual(steps, blocks)\\n                self.assertEqual(acts.shape[1], frames.shape[1] - 1 + blocks)\\n            # Context must contain actual previously executed actions.\\n            torch.testing.assert_close(agent.encoder.calls[1][1][0, 0], agent.executed[0])\\n            torch.testing.assert_close(agent.encoder.calls[2][1][0, :2], torch.stack(agent.executed[:2]))\\n\\n    def test_horizon_changes_candidate_selection(self):\\n        short, long = make_agent(4), make_agent(16)\\n        self.assertGreater(float(short.step(np.array([0], np.uint8))[0, 0]), 0)\\n        self.assertLess(float(long.step(np.array([0], np.uint8))[0, 0]), 0)\\n\\n    def test_default_matches_explicit_eight_seconds(self):\\n        default, explicit = make_agent(), make_agent(16)\\n        for _ in range(5):\\n            np.testing.assert_array_equal(default.step(np.array([0], np.uint8)),\\n                                          explicit.step(np.array([0], np.uint8)))\\n            torch.testing.assert_close(default.last_distance, explicit.last_distance)\\n\\n    def test_invalid_horizons(self):\\n        for blocks in (0, -1, 17, 4.5, True):\\n            with self.assertRaises(ValueError):\\n                make_agent(blocks)\\n\\n    def test_real_observations_alone_advance_goals(self):\\n        agent = make_agent(4)\\n        agent.reset(np.array([[1], [2]], np.uint8), np.array([0], np.uint8))\\n        agent.step(np.array([0], np.uint8))\\n        self.assertEqual(agent.index, 0)  # Imagined candidate reaches goal 1.\\n        agent.step(np.array([1], np.uint8))\\n        self.assertEqual(agent.index, 1)\\n\\n\\nif __name__ == \'__main__\':\\n    unittest.main()\\n"}')
for relative, source in sources.items():
    target = ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source)
import assignment.training.planning.agent as agent_module
importlib.reload(agent_module)
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                       '-p', 'test_planning_horizon.py', '-v'], cwd=ROOT)
from assignment.planning.horizon_experiment import run_experiment, summarize
print('Experiment source installed; horizon tests passed.')


In [ ]:
# Real-model regression: explicit 8-second prediction must match the official class.
import numpy as np
reference_scope = {}
exec(compile('import numpy as np\nimport torch\nfrom assignment.training.data.cache import RAW_ACTION_DIM, STRIDE\nfrom assignment.training.planning.agent import LatentPolicyAgent\n\n\nclass GoalPlanningAgent(LatentPolicyAgent):\n    """Baseline test-time planner over policy action chunks using the latent world model.\n\n    Edit this class to experiment with your own cost functions, rollout horizons,\n    candidate refinement rules, or subgoal switching criteria.\n    """\n\n    def __init__(self, policy_path, device="cuda", candidates=16, tau=0.25, **kwargs):\n        super().__init__(policy_path, device, **kwargs)\n        self.candidates, self.tau = candidates, tau\n        self.goals = None\n\n    def reset(self, goal_frames=None, init_frame=None, seed=None):\n        if goal_frames is not None and init_frame is None:\n            raise ValueError("init_frame is required with goal_frames")\n        super().reset(seed)\n        self.frames, self.executed, self.index = [], [], 0\n        self.last_distance = self.goals = self.gap = None\n        if goal_frames is not None:\n            # Encode all M subgoal frames (M, 128, 256, 3) -> (M, 192).\n            self.goals = self._encode(np.asarray(goal_frames))\n            prev = torch.cat([self._encode(np.asarray(init_frame)[None]), self.goals[:-1]])\n            self.gap = (self.goals - prev).square().sum(-1)\n\n    def goal_index(self, z):\n        # Advance monotonically while ||z_k - g_m||_2^2 < tau * ||g_m - g_{m-1}||_2^2.\n        while self.index < len(self.goals) - 1 and \\\n                (z - self.goals[self.index]).square().sum() < self.tau * self.gap[self.index]:\n            self.index += 1\n        return self.index\n\n    @torch.no_grad()\n    def _encode(self, frames):\n        x = torch.from_numpy(np.ascontiguousarray(frames)).to(self.device)[:, None]\n        with self._autocast():\n            return self.encoder.encode(x)[:, 0].float()\n\n    @torch.no_grad()\n    def step(self, frame):\n        if self.goals is None:\n            raise RuntimeError("Call agent.reset(goal_frames, init_frame) before step().")\n        # 1. Encode the current (128, 256, 3) frame into z_k in R^192 and update subgoal index.\n        z = self._encode(frame[None])[0]\n        ctx = self.encoder.cfg.context\n        self.frames = (self.frames + [z])[-ctx:]\n        self.history = (self.history + [(z - self.z_mean) / self.z_std])[-self.policy.cfg.context:]\n        goal = self.goals[self.goal_index(z)]\n\n        if not self.queue:\n            # 2. Sample N = self.candidates action chunks of shape (N, generate, 125).\n            pctx = [self.history[0]] * (self.policy.cfg.context - len(self.history)) + self.history\n            n = self.candidates\n            blocks = self.policy.sample(torch.stack(pctx)[None].expand(n, -1, -1), self.generate,\n                                        self.n_steps, self.lag, self.generator)\n            best = 0\n            if n > 1:\n                # 3. Roll all N candidates forward in the world model from the last C<=3 frames.\n                past = self.executed[len(self.executed) - len(self.frames) + 1:] if len(self.frames) > 1 else []\n                acts = torch.cat([torch.stack(past)[None].expand(n, -1, -1), blocks], 1) if past else blocks\n                with self._autocast():\n                    pred = self.encoder.rollout(torch.stack(self.frames)[None].expand(n, -1, -1),\n                                                self.encoder.action_encoder(acts), self.generate)\n                # 4. Pick the candidate whose final predicted latent is closest to the active subgoal.\n                self.last_distance = (pred[:, -1].float() - goal).square().sum(-1)\n                best = int(self.last_distance.argmin())\n            raw = np.clip(blocks[best, :self.execute].cpu().numpy() * self.a_std + self.a_mean, -1, 1)\n            self.queue = list(raw.reshape(-1, STRIDE, RAW_ACTION_DIM))\n\n        block = self.queue.pop(0)\n        self.executed.append(torch.from_numpy((block.reshape(-1) - self.a_mean) / self.a_std).to(self.device))\n        return block', '<official-notebook-planner>', 'exec'), reference_scope)
OfficialGoalPlanningAgent = reference_scope['GoalPlanningAgent']
common = dict(world_model=str(WM), candidates=16, generate=16, execute=1, n_steps=10, lag=0, tau=.25)
official = OfficialGoalPlanningAgent(str(POLICY), 'cuda', **common)
ours = agent_module.GoalPlanningAgent(str(POLICY), 'cuda', planning_horizon=16, **common)
with np.load(GOALS) as d:
    i = list(d['seeds']).index(100038)
    reference_goals = d['sub_frames'][i, :int(d['sub_count'][i])].copy()
    reference_init = d['init_frames'][i].copy()
for a in (official, ours):
    a.reset(reference_goals, reference_init, seed=100038)
for observation in [reference_init, *reference_goals[:3]]:
    np.testing.assert_array_equal(official.step(observation), ours.step(observation))
    np.testing.assert_array_equal(official.last_distance.cpu().numpy(), ours.last_distance.cpu().numpy())
del official, ours
torch.cuda.empty_cache()
print('PASS: 8-second actions and candidate costs exactly match the original planner on 4 observations.')


## Configure the comparison

Default: one scene × one seed × three horizons = three full episodes.
Use `EPISODE_STEPS = 40` only for a quick implementation check, with a fresh output folder.
For a later development comparison, predeclare matching validation scenes and use seeds [0, 2, 3].
Keep `PLANNING_SECONDS = [2, 4, 8]`, policy generation and official goal images fixed.


In [ ]:
from datetime import datetime, timezone
SCENES = [100038]
SAMPLE_SEEDS = [0]
PLANNING_SECONDS = [2, 4, 8]
EPISODE_STEPS = 1795
OUT = Path('/content/horizon-results') / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
print('Results folder:', OUT)


In [ ]:
# A brief end-to-end check of all horizons before the full experiment.
smoke_rows = run_experiment(POLICY, WM, GOALS, OUT / 'smoke', scenes=[100038],
                            sample_seeds=[0], episode_steps=40)
print('All 3 short smoke runs passed; these are not performance results.')


In [ ]:
rows = run_experiment(
    POLICY, WM, GOALS, OUT / 'full', scenes=SCENES, sample_seeds=SAMPLE_SEEDS,
    planning_seconds=PLANNING_SECONDS, episode_steps=EPISODE_STEPS,
)


In [ ]:
from IPython.display import Video, display, Markdown
groups = summarize(rows)
table = '| Horizon | Episodes | Mean peak | Mean final | Success | Mean decision ms |\n'
table += '|---|---:|---:|---:|---:|---:|\n'
for r in groups:
    table += f"| {r['planning_seconds']}s | {r['episodes']} | {r['mean_peak_boxes']:.2f} | {r['mean_final_boxes']:.2f} | {r['success_rate']:.1%} | {r['mean_decision_ms']:.1f} |\n"
display(Markdown(table))
if EPISODE_STEPS != 1795:
    display(Markdown('**Short smoke test only: do not interpret box counts as performance.**'))
for row in rows:
    display(Markdown(f"**{row['planning_seconds']}s horizon — scene {row['scene']}, seed {row['sample_seed']}**"))
    display(Video(row['video'], embed=True, width=512))


## Save your evidence

Run the cell below to download a ZIP containing videos, per-decision traces, model/goal
SHA256 hashes, source hashes, settings, episode results and the summary. Colab files
are temporary until you download them. A rerun with identical settings and `OUT` resumes
completed episodes; changing settings/source requires a fresh output folder.

Questions for later analysis: does a shorter horizon improve box counts, reduce decision
time, or become too shortsighted to reach a distant subgoal? One scene cannot establish
an improvement. Compare matching scenes/seeds and keep development and confirmation separate.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
files.download(archive)
